<a href="https://colab.research.google.com/github/Physalis-Alkekengi/ASIGROMACS/blob/main/PostMDData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Catalytic Proximity Analysis and Trajectory Assessment
import subprocess
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Dependency Management: MDTraj for trajectory analysis
try:
    import mdtraj as md
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "mdtraj"], stdout=subprocess.DEVNULL)
    import mdtraj as md

print("Initiating reaction coordinate analysis...")

# 1. DATA RETRIEVAL AND EXTRACTION
zip_file = "FINAL_SIMULATION_DATA.zip"
if os.path.exists(zip_file):
    print(f"Extracting simulation data: {zip_file}")
    subprocess.run(f"unzip -o {zip_file}", shell=True, stdout=subprocess.DEVNULL)

# 2. TRAJECTORY LOADING
# Critical: Topology (.gro) and Trajectory (.xtc) must have identical atom counts.
# We prioritize the production output which includes solvent.
gro_file = "step5_production.gro"
xtc_file = "step5_production.xtc"

# Fallback mechanism for trajectory mismatch
if not os.path.exists(xtc_file):
    print("Warning: Raw trajectory not found. Defaulting to post-processed trajectory.")
    xtc_file = "final_movie_no_water.xtc"

if os.path.exists(gro_file) and os.path.exists(xtc_file):
    print(f"Loading Trajectory: {xtc_file}")
    print(f"Loading Topology:   {gro_file}")

    try:
        traj = md.load(xtc_file, top=gro_file)
        print(f"Frames loaded: {traj.n_frames}")

        # 3. ATOM SELECTION (DSL)
        # Target: Polypropylene Ligand
        ligand_indices = traj.topology.select("resname LIG or resname UNK")

        # Target: Catalytic Nucleophiles/Bases (Sidechain atoms only)
        # Broad selection for CYS, SER, TYR, HSD to identify nearest active residue
        catalytic_indices = traj.topology.select("protein and (resname CYS or resname SER or resname TYR or resname HSD) and sidechain")

        if len(ligand_indices) == 0:
            print("Error: Ligand (LIG/UNK) not detected in topology.")
        else:
            # 4. GEOMETRIC CALCULATION
            print("Computing minimum pairwise distances...")

            # Calculate distance between every ligand atom and every catalytic sidechain atom
            # scheme='closest' returns the shortest distance per frame
            # min_distances shape: (n_frames, 1)
            min_distances = md.compute_contacts(traj, contacts=[(i, j) for i in ligand_indices for j in catalytic_indices], scheme='closest')[0]

            # Convert nanometers to Angstroms (1 nm = 10 A)
            distances_angstrom = min_distances[:, 0] * 10
            avg_dist = np.mean(distances_angstrom)
            min_dist = np.min(distances_angstrom)

            print("\n" + "="*50)
            print(f"GEOMETRIC ANALYSIS RESULTS")
            print(f"Average Interaction Distance: {avg_dist:.2f} \u00C5") # \u00C5 is Angstrom symbol
            print(f"Minimum Approach Distance:    {min_dist:.2f} \u00C5")
            print("="*50)

            # 5. KINETIC FEASIBILITY ASSESSMENT
            print("INTERPRETATION:")
            if avg_dist < 3.5:
                print("   Status: Near-Attack Conformation (NAC) Achieved (< 3.5 \u00C5).")
                print("   Assessment: High probability of catalytic turnover.")
            elif avg_dist < 5.0:
                print("   Status: Proximity Bound (3.5 - 5.0 \u00C5).")
                print("   Assessment: Substrate bound but requires conformational fluctuation for catalysis.")
            else:
                print("   Status: Non-Reactive (> 5.0 \u00C5).")
                print("   Assessment: Steric hindrance or weak affinity detected. Mutation required.")

            # 6. DATA VISUALIZATION
            plt.figure(figsize=(10, 6))
            plt.plot(traj.time, distances_angstrom, color='#8B0000', linewidth=1.5, label='Ligand-Nucleophile Distance')

            # Thresholds
            plt.axhline(y=3.5, color='green', linestyle='--', linewidth=2, label="Catalytic Cutoff (3.5 \u00C5)")
            plt.axhline(y=5.0, color='orange', linestyle=':', linewidth=2, label="Interaction Threshold (5.0 \u00C5)")

            plt.xlabel("Simulation Time (ps)", fontsize=12)
            plt.ylabel(r"Distance ($\AA$)", fontsize=12)
            plt.title("Reaction Coordinate Trajectory", fontsize=14, fontweight='bold')
            plt.legend(loc='upper right')
            plt.grid(True, alpha=0.4, linestyle='--')

            # Save and display
            plt.tight_layout()
            plt.savefig("catalytic_distance_plot.png", dpi=300)
            plt.show()

    except ValueError as e:
        print("\nCRITICAL TOPOLOGY ERROR:")
        print("Atom count mismatch detected between .gro (topology) and .xtc (trajectory).")
        print("Diagnostic: Ensure the topology file includes solvent if using the raw production trajectory.")
        print(f"System Message: {e}")

else:
    print("Error: Input files (gro/xtc) not located. Verify archive extraction.")

In [ ]:
# @title Interaction Energy Quantification and Ligand Verification
import os
import subprocess
import pandas as pd
import sys
import numpy as np

# Configuration
LIGAND_RESNAME = "LIG"
print(f"Initiating topology verification for residue: {LIGAND_RESNAME}")

# 1. ENVIRONMENT VALIDATION
# Ensure standard GROMACS binaries are available for analysis tools
gmx_cmd = "/usr/bin/gmx"
if not os.path.exists(gmx_cmd):
    print("Installing GROMACS analysis toolkit...")
    subprocess.run("apt-get update > /dev/null", shell=True)
    subprocess.run("apt-get install -y gromacs > /dev/null", shell=True)

# 2. INDEX GENERATION AND GROUP VALIDATION
# We attempt to create an index group for the ligand. Failure here indicates a topology error.
print("Generating index groups...")
if os.path.exists("step5_production.gro"):
    # Command: 'ri LIG' (Select residue index LIG) -> 'q' (Save/Quit)
    cmd = [gmx_cmd, "make_ndx", "-f", "step5_production.gro", "-o", "index.ndx"]

    # Send input stream to stdin
    process = subprocess.run(cmd, input=f"ri {LIGAND_RESNAME}\nq\n".encode(), capture_output=True)

    if os.path.exists("index.ndx") and b" 0 atoms" not in process.stdout:
        print("   Index generation successful. Ligand group verified.")
    else:
        print("CRITICAL ERROR: Ligand group generation failed.")
        print("   Diagnostic: The residue 'LIG' does not exist in the coordinate file.")
        print("   Action: Abort analysis. Re-evalute system setup (Step 1).")
        sys.exit()
else:
    print("Error: Input coordinates ('step5_production.gro') not found.")
    sys.exit()

# 3. INTERACTION ENERGY CALCULATION (RERUN PROTOCOL)
# We reconstruct the energy file (.edr) using the existing trajectory but with specific energy groups defined.
print("Executing energy recalculation (Rerun)...")

rerun_mdp_content = f"""
integrator = md
nsteps = -1       ; Process all frames
dt = 0.002
continuation = yes
constraints = h-bonds
ns_type = grid
cutoff-scheme = Verlet
coulombtype = PME
energygrps = Protein {LIGAND_RESNAME}  ; Explicitly calculate short-range energies between these groups
"""

with open("rerun_energy.mdp", "w") as f:
    f.write(rerun_mdp_content)

# Pre-processing (grompp)
# Note: We ignore warnings (-maxwarn 2) often caused by mismatched mdp parameters in reruns
subprocess.run(f"{gmx_cmd} grompp -f rerun_energy.mdp -c step5_production.gro -p topol.top -n index.ndx -o energy_rerun.tpr -maxwarn 2", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Execution (mdrun -rerun)
subprocess.run(f"{gmx_cmd} mdrun -rerun step5_production.xtc -s energy_rerun.tpr -e interaction.edr", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Energy Extraction (gmx energy)
# Extract Coulombic (SR) and Lennard-Jones (SR) terms
selection_terms = f"Coul-SR:Protein-{LIGAND_RESNAME} LJ-SR:Protein-{LIGAND_RESNAME}"
subprocess.run(f"echo {selection_terms} 0 | {gmx_cmd} energy -f interaction.edr -o binding_score.xvg", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 4. QUANTITATIVE ASSESSMENT
if os.path.exists("binding_score.xvg"):
    data_points = []
    try:
        with open("binding_score.xvg", 'r') as f:
            for line in f:
                if not line.startswith(("#", "@")):
                    # Columns: Time, Coul-SR, LJ-SR
                    data_points.append([float(x) for x in line.split()])

        if data_points:
            df = pd.DataFrame(data_points, columns=["Time", "Coul-SR", "LJ-SR"])

            # Calculate Total Interaction Energy ($E_{int} = E_{Coul} + E_{LJ}$)
            df["Total"] = df["Coul-SR"] + df["LJ-SR"]
            mean_score = df["Total"].mean()
            std_score = df["Total"].std()

            print("\n" + "="*50)
            print(f"INTERACTION ENERGY ANALYSIS")
            print(f"Mean Interaction Energy: {mean_score:.2f} +/- {std_score:.2f} kJ/mol")
            print("="*50)

            # Threshold Analysis
            # A score near 0 kJ/mol implies no physical contact (infinite separation).
            if abs(mean_score) < 5.0:
                print("DIAGNOSTIC: NULL INTERACTION DETECTED.")
                print("   Observation: Interaction energy is negligible (~0 kJ/mol).")
                print("   Conclusion: The ligand is physically dissociated or non-existent in the binding pocket.")
                print("   Recommendation: Perform rigid-body docking (Autodock Vina) before MD.")
            else:
                print("DIAGNOSTIC: STABLE COMPLEX VERIFIED.")
                print("   Observation: Significant attractive/repulsive forces detected.")
                print("   Conclusion: Ligand is physically engaged with the protein.")
        else:
            print("Error: Empty dataset derived from energy file.")
    except Exception as e:
        print(f"Data Parsing Error: {e}")
else:
    print("Error: Energy calculation failed. Verify group definitions in index.ndx.")

In [ ]:
# @title System Composition Audit and Residue Enumeration
import os

print("Initiating molecular inventory scan...")

# Target Coordinate File
gro_file = "step5_production.gro"

if os.path.exists(gro_file):
    residues = set()

    # 1. PARSE COORDINATE FILE
    # GROMOS87 format: Header (2 lines), Atoms, Box Vectors (1 line)
    with open(gro_file, "r") as f:
        lines = f.readlines()
        # Extract residue names from atom lines (Indices 2 to -1)
        for line in lines[2:-1]:
            # Standard Fixed Column Width: Residue Name is cols 5-10
            res_name = line[5:10].strip()
            residues.add(res_name)

    print(f"Unique molecular species detected: {len(residues)}")
    print("-" * 50)

    # 2. CLASSIFICATION AND REPORTING
    sorted_residues = sorted(list(residues))

    # Standard Forcefield Nomenclatures
    solvent_codes = ["SOL", "TIP3", "HOH", "WAT"]
    ion_codes = ["NA", "CL", "K", "SOD", "CLA", "MG", "ZN", "CAL"]
    cofactor_codes = ["GSH", "HEM", "NAD", "NADP"]

    # Heuristic for detecting potential non-protein ligands
    plastic_candidates = []

    for r in sorted_residues:
        description = "Unclassified"

        if r in solvent_codes:
            description = "Solvent (Water)"
        elif r in ion_codes:
            description = "Counter-Ion"
        elif r in cofactor_codes:
            description = "Cofactor"
        elif len(r) == 3 and r not in cofactor_codes:
            description = "Standard Residue (Likely Protein)"
        else:
            description = "Non-Standard Residue (Ligand Candidate)"
            plastic_candidates.append(r)

        # Override for common ligand names used in tutorials if length is 3 (e.g., LIG, I01)
        if r in ["LIG", "UNK", "I01", "PP"]:
             description = "Substrate / Ligand"
             if r not in plastic_candidates: plastic_candidates.append(r)

        print(f"   - {r:<10} | {description}")

    print("-" * 50)

    # 3. SYSTEM STATE ASSESSMENT
    if len(plastic_candidates) == 0:
        print("\nDIAGNOSTIC: APO STATE DETECTED.")
        print("   Observation: System contains only Enzyme, Solvent, and Ions.")
        print("   Assessment: No polymer substrate detected.")
        print("   Action Required: Execute Molecular Docking protocol to insert ligand.")
    else:
        print(f"\nDIAGNOSTIC: HOLO STATE DETECTED.")
        print(f"   Ligand Candidates: {plastic_candidates}")
        print("   Assessment: System ready for production dynamics.")

else:
    print(f"Error: Input file '{gro_file}' not found. Verify simulation completion.")

In [ ]:
# @title Atomic-Level Constituent Analysis
import os
import subprocess

print("Initiating atomic inventory scan...")

# Target Coordinate File
gro_file = "step5_production.gro"

# 1. FILE INTEGRITY CHECK & RECOVERY
if not os.path.exists(gro_file):
    # Attempt extraction from archive if primary file is missing
    if os.path.exists("FINAL_SIMULATION_DATA.zip"):
        print("Restoring data from archive...")
        subprocess.run("unzip -o FINAL_SIMULATION_DATA.zip", shell=True, stdout=subprocess.DEVNULL)

if os.path.exists(gro_file):
    print(f"Coordinate file loaded: {gro_file}")

    # 2. PARSING ROUTINE
    with open(gro_file, 'r') as f:
        lines = f.readlines()

    print(f"Total atomic entries: {len(lines) - 3}") # Subtract header/footer
    print("\n" + "="*60)
    print(f"{'Residue':<10} | {'Atom Name':<10} | {'Count':<10} | {'Classification'}")
    print("="*60)

    inventory = {}

    # 3. ITERATIVE INSPECTION
    # Skip header (2 lines) and box vectors (last line)
    for line in lines[2:-1]:
        # GROMOS87 Fixed Width Format:
        # Residue Name: Indices 5-10
        # Atom Name:    Indices 10-15
        res_name = line[5:10].strip()
        atom_name = line[10:15].strip()

        # Composite Key for unique identification
        key = (res_name, atom_name)
        if key not in inventory:
            inventory[key] = 0
        inventory[key] += 1

    # 4. CLASSIFICATION LOGIC
    # Standard Amino Acid Library (Exclusion List)
    standard_aa = {
        "ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY","HIS","ILE",
        "LEU","LYS","MET","PHE","PRO","SER","THR","TRP","TYR","VAL"
    }

    solvent_res = {"SOL", "TIP3", "HOH", "WAT"}
    ion_res     = {"NA", "CL", "K", "SOD", "CLA", "POT", "ZN", "MG", "CAL"}

    for (res, atom), count in sorted(inventory.items()):
        classification = "Unknown / Non-Standard"

        # Filter: Skip standard protein residues to reduce noise
        if res in standard_aa:
            continue

        # Classification Hierarchy
        if res in solvent_res:
            classification = "Solvent (Water)"
        elif res in ion_res:
            classification = "Counter-Ion"
        elif res == "GSH":
            classification = "Cofactor (Glutathione)"
        elif "LIG" in res or "UNK" in res or "PP" in res:
             # Atomic Element Check
             if "C" in atom:
                 classification = "Carbon-Based Ligand (Confirmed)"
             elif "O" in atom or "H" in atom:
                 classification = "Ligand (Potential Mislabeled Water)"
             else:
                 classification = "Ligand (Heteroatom)"

        # Output relevant entries
        print(f"{res:<10} | {atom:<10} | {count:<10} | {classification}")

    print("="*60)
    print("\nDIAGNOSTIC REPORT:")
    print("1. Ion Check: Verify 'CLA/SOD' entries are distinct from ligand atoms.")
    print("2. Ligand Integrity: Inspect 'LIG' atom names.")
    print("   - Presence of Carbon ('C') confirms polymeric/organic nature.")
    print("   - Exclusive Oxygen/Hydrogen content suggests solvent misclassification.")

else:
    print("Error: Input coordinate file unavailable. Verify archive extraction.")